### Import Libraries

In [38]:
import requests
import numpy as np
import glob
import pandas as pd
import sqlite3
from datetime import datetime
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET

url = 'https://web.archive.org/web/20230908091635/https://en.wikipedia.org/wiki/List_of_largest_banks'
table_attribs = ['Name', 'MC_USD_Billion']
csv_path = './exchange_rate.csv'
output_path = 'largest_banks_data.csv'
db_name = 'Banks.db'
table_name = 'Largest_banks'

### Task 1: Logging function

In [39]:
def log_progress(message):
    timestamp_format = '%Y-%b-%d-%H:%M:%S'
    now = datetime.now()
    timestamp = now.strftime(timestamp_format)

    with open('./code_log.txt', 'a') as f:
        f.write(timestamp + ' : ' + message + '\n')

### Task 2 : Extraction of data

In [40]:
def extract(url, table_attribs):
    page = requests.get(url).text
    soup = BeautifulSoup(page, 'html.parser')
    df = pd.DataFrame(columns=table_attribs)
    tables = soup.find_all('tbody')
    rows = tables[0].find_all('tr')

    for row in rows:
        col = row.find_all('td')
        if len(col)!=0:
            data_dict = {table_attribs[0]: col[1].find_all('a')[-1].contents[0],
                        table_attribs[1]: float(col[2].contents[0].strip())}
            df1 = pd.DataFrame(data_dict, index=[0])
            df = pd.concat([df, df1], ignore_index=True)
    
    return df

### Task 3 : Transformation of data

In [41]:
def transform(df, csv_path):
    er = pd.read_csv(csv_path).set_index('Currency').to_dict()['Rate']
    df['MC_GBP_Billion'] = [np.round(x * er['GBP'],2) for x in df['MC_USD_Billion']]
    df['MC_EUR_Billion'] = [np.round(x * er['EUR'],2) for x in df['MC_USD_Billion']]
    df['MC_INR_Billion'] = [np.round(x * er['INR'],2) for x in df['MC_USD_Billion']]
    return df

### Task 4: Loading to CSV and Database

In [42]:
def load_to_csv(df, output_path):
    df.to_csv(output_path)

def load_to_db(df, sql_connection, table_name):
    df.to_sql(table_name, sql_connection, if_exists='replace', index=False)

### Task 5: Function to Run queries on Database

In [43]:
def run_query(query_statement, sql_connection):
    print(query_statement)
    query_output = pd.read_sql(query_statement, sql_connection)
    print(query_output) 

In [45]:
log_progress('Preliminaries Complete. Initiating ETL Process.')
df = extract(url, table_attribs)
log_progress('Data Extraction Complete. Initiating Transformation Process.')

df = transform(df, csv_path)
log_progress('Data Transformation Complete. Initiating Loading Process.')

load_to_csv(df, output_path=output_path)
log_progress('Data Saved to CSV files.')

sql_connection = sqlite3.connect(db_name)
log_progress('SQL Connection Initiated.')

load_to_db(df, sql_connection, table_name)
log_progress('Data Loaded to Database as a table, Executing queries.')

query_statement = f"SELECT * FROM {table_name}"
run_query(query_statement, sql_connection)

query_statement = f"SELECT AVG(MC_GBP_Billion) FROM {table_name}"
run_query(query_statement, sql_connection)

query_statement = f"SELECT Name FROM {table_name} LIMIT 5"
run_query(query_statement, sql_connection) 

log_progress('Process Complete.')
sql_connection.close()
log_progress('Server Connection Closed.')


SELECT * FROM Largest_banks
                                      Name  MC_USD_Billion  MC_GBP_Billion  \
0                           JPMorgan Chase          432.92          346.34   
1                          Bank of America          231.52          185.22   
2  Industrial and Commercial Bank of China          194.56          155.65   
3               Agricultural Bank of China          160.68          128.54   
4                                HDFC Bank          157.91          126.33   
5                              Wells Fargo          155.87          124.70   
6                        HSBC Holdings PLC          148.90          119.12   
7                           Morgan Stanley          140.83          112.66   
8                  China Construction Bank          139.82          111.86   
9                            Bank of China          136.81          109.45   

   MC_EUR_Billion  MC_INR_Billion  
0          402.62        35910.71  
1          215.31        19204.58  
2    